# Lesson 3 — Matrix Multiplication & Tensor Shapes

## 学习目标

这一节的核心目标不是记忆 `torch.matmul()` 的规则，而是能够看到矩阵乘法时直接推导 Tensor shape。

完成本节后，应能够：

1. 理解普通二维矩阵乘法；
2. 理解矩阵乘法中的 inner dimension 和 outer dimensions；
3. 推导 `(B,T,D) @ (D,H) -> (B,T,H)`；
4. 理解 PyTorch 对高维 Tensor 的 batched matrix multiplication；
5. 理解为什么 Attention 中需要转置 K；
6. 推导：

$$
QK^\top
$$

从

$$
Q.shape=(B,H,T,D_h)
$$

和

$$
K.shape=(B,H,T,D_h)
$$

得到：

$$
QK^\top.shape=(B,H,T,T)
$$

7. 推导 Attention Scores 与 V 相乘后的 shape；
8. 开始从 Tensor shape 的角度理解 Self-Attention。


## 1. 二维矩阵乘法

最基础的矩阵乘法为：

$$
A \in \mathbb{R}^{M \times K}
$$

$$
B \in \mathbb{R}^{K \times N}
$$

那么：

$$
C=AB
$$

满足：

$$
C \in \mathbb{R}^{M \times N}
$$

也就是：

$$
(M,K)(K,N)\rightarrow(M,N)
$$

最重要的观察是：

> 中间两个维度 K 必须相同，并且在结果中消失。

可以简单记成：

$$
(M,\cancel{K})(\cancel{K},N)
\rightarrow
(M,N)
$$

这里的 K 常被称为 contraction dimension 或 inner dimension。


In [1]:
import torch

A = torch.randn(3, 4)
B = torch.randn(4, 5)

C = A @ B

print("A:", A.shape)
print("B:", B.shape)
print("C:", C.shape)


A: torch.Size([3, 4])
B: torch.Size([4, 5])
C: torch.Size([3, 5])


## 2. 为什么 Inner Dimension 必须相同？

考虑：

$$
A.shape=(3,4)
$$

$$
B.shape=(4,5)
$$

矩阵 A 的每一行包含 4 个元素。

矩阵 B 的每一列也包含 4 个元素。

计算结果中的一个元素：

$$
C_{ij}
$$

本质上是：

$$
C_{ij}
=
\sum_{k=1}^{4}
A_{ik}B_{kj}
$$

也就是：

> A 的一行与 B 的一列进行 dot product。

因此二者长度必须相同。

所以：

$$
(3,4)(4,5)
$$

可以相乘。

但是：

$$
(3,4)(3,5)
$$

不能相乘，因为：

$$
4\neq3
$$


In [2]:
A = torch.randn(3, 4)
B = torch.randn(3, 5)

try:
    C = A @ B
except RuntimeError as error:
    print(error)


mat1 and mat2 shapes cannot be multiplied (3x4 and 3x5)


## 3. Matrix Multiplication vs Element-wise Multiplication

PyTorch 中：

`A * B`

表示逐元素乘法。

而：

`A @ B`

表示矩阵乘法。

这是两种完全不同的运算。

例如：

$$
A.shape=(3,4)
$$

$$
B.shape=(4,5)
$$

可以执行：

$$
A@B
$$

但不能直接执行：

$$
A*B
$$

因为 `(3,4)` 和 `(4,5)` 无法按 broadcasting 规则进行逐元素运算。

因此以后看到：

`*`

应该想到：

> element-wise operation

看到：

`@`

应该想到：

> matrix multiplication


In [3]:
A = torch.randn(3, 4)
B = torch.randn(4, 5)

C = A @ B

print("Matrix multiplication:")
print(C.shape)

try:
    D = A * B
except RuntimeError as error:
    print("\nElement-wise multiplication failed:")
    print(error)


Matrix multiplication:
torch.Size([3, 5])

Element-wise multiplication failed:
The size of tensor a (4) must match the size of tensor b (5) at non-singleton dimension 1


## 4. Transformer 中最常见的矩阵乘法：Linear Projection

现在开始进入 Transformer 中真正使用的 shape。

假设输入：

$$
X.shape=(B,T,D)
$$

其中：

- $B$：Batch Size
- $T$：Sequence Length
- $D$：Input / Model Dimension

权重矩阵：

$$
W.shape=(D,H)
$$

那么：

$$
Y=XW
$$

输出：

$$
Y.shape=(B,T,H)
$$

也就是：

$$
(B,T,D)(D,H)
\rightarrow
(B,T,H)
$$

可以把最后两个维度单独看：

$$
(T,D)(D,H)
\rightarrow
(T,H)
$$

而 Batch 维 B 被保留下来。


In [4]:
B = 2
T = 3
D = 4
H = 6

x = torch.randn(B, T, D)
weight = torch.randn(D, H)

y = x @ weight

print("x     :", x.shape)
print("weight:", weight.shape)
print("y     :", y.shape)


x     : torch.Size([2, 3, 4])
weight: torch.Size([4, 6])
y     : torch.Size([2, 3, 6])


## 5. 高维 Tensor 的 Matmul

对于：

$$
x.shape=(B,T,D)
$$

和：

$$
W.shape=(D,H)
$$

PyTorch 不会把整个 `(B,T,D)` 当成一个二维矩阵。

矩阵乘法主要作用于最后的矩阵维度：

$$
D
$$

和：

$$
(D,H)
$$

进行 contraction。

因此：

$$
(B,T,D)(D,H)
$$

可以理解为：

$$
(B,T,\cancel{D})(\cancel{D},H)
$$

得到：

$$
(B,T,H)
$$

B 和 T 都被保留下来。

从神经网络的角度看：

> 同一个 Linear Transformation 被独立应用到所有 batch 中的所有 token。


## 6. 从单个 Token 理解 Linear Projection

假设某个 token 表示为：

$$
x_{token}\in\mathbb{R}^{D}
$$

权重：

$$
W\in\mathbb{R}^{D\times H}
$$

那么：

$$
x_{token}W
$$

得到：

$$
\mathbb{R}^{H}
$$

也就是：

$$
(D)(D,H)\rightarrow(H)
$$

对于整个 sequence：

$$
(T,D)(D,H)
\rightarrow
(T,H)
$$

再加上 Batch：

$$
(B,T,D)(D,H)
\rightarrow
(B,T,H)
$$

因此 Linear Layer 可以理解为：

> 对每一个 token 的 D 维表示应用同一个线性变换。


In [5]:
B = 2
T = 3
D = 4
H = 6

x = torch.randn(B, T, D)
weight = torch.randn(D, H)

y_full = x @ weight

y_manual = torch.empty(B, T, H)

for b in range(B):
    for t in range(T):
        y_manual[b, t] = x[b, t] @ weight

print(torch.allclose(y_full, y_manual))


True


## 7. Linear Layer = Matrix Multiplication + Broadcasting

Linear Layer 通常为：

$$
Y=XW+b
$$

其中：

$$
X.shape=(B,T,D)
$$

$$
W.shape=(D,H)
$$

$$
b.shape=(H,)
$$

第一步：

$$
XW
$$

得到：

$$
(B,T,H)
$$

第二步：

$$
(B,T,H)+(H,)
$$

利用上一节学习的 Broadcasting：

$$
(H,)
\rightarrow
(1,1,H)
$$

于是：

$$
(B,T,H)+(1,1,H)
\rightarrow
(B,T,H)
$$

这正好把 Lesson 2 的 Broadcasting 和本节的 Matrix Multiplication 连接起来。


In [6]:
B = 2
T = 3
D = 4
H = 6

x = torch.randn(B, T, D)
weight = torch.randn(D, H)
bias = torch.randn(H)

y = x @ weight + bias

print("x     :", x.shape)
print("weight:", weight.shape)
print("bias  :", bias.shape)
print("y     :", y.shape)


x     : torch.Size([2, 3, 4])
weight: torch.Size([4, 6])
bias  : torch.Size([6])
y     : torch.Size([2, 3, 6])


## 8. Batched Matrix Multiplication

对于高维 Tensor，PyTorch 将最后两个维度视为矩阵维度。

前面的维度则可以理解为 batch dimensions。

例如：

$$
A.shape=(2,3,4,5)
$$

$$
B.shape=(2,3,5,6)
$$

最后两个维度：

$$
(4,5)(5,6)
\rightarrow
(4,6)
$$

前面的：

$$
(2,3)
$$

保持不变。

所以：

$$
(2,3,4,5)(2,3,5,6)
\rightarrow
(2,3,4,6)
$$

一般形式：

$$
(...,M,K)(...,K,N)
\rightarrow
(...,M,N)
$$

其中 `...` 表示 batch dimensions。


In [7]:
A = torch.randn(2, 3, 4, 5)
B = torch.randn(2, 3, 5, 6)

C = A @ B

print("A:", A.shape)
print("B:", B.shape)
print("C:", C.shape)


A: torch.Size([2, 3, 4, 5])
B: torch.Size([2, 3, 5, 6])
C: torch.Size([2, 3, 4, 6])


## 9. Attention 中的 Q、K、V

现在正式进入 Self-Attention 的核心 shape。

Multi-Head Attention 中通常有：

$$
Q.shape=(B,H,T,D_h)
$$

$$
K.shape=(B,H,T,D_h)
$$

$$
V.shape=(B,H,T,D_h)
$$

其中：

- $B$：Batch Size
- $H$：Number of Heads
- $T$：Sequence Length
- $D_h$：Head Dimension

前一节已经看到：

$$
(B,T,D)
\rightarrow
(B,T,H,D_h)
\rightarrow
(B,H,T,D_h)
$$

所以现在可以把 Q、K、V 都看成：

$$
(B,H,T,D_h)
$$


In [8]:
B = 2
H = 4
T = 8
Dh = 16

q = torch.randn(B, H, T, Dh)
k = torch.randn(B, H, T, Dh)
v = torch.randn(B, H, T, Dh)

print("Q:", q.shape)
print("K:", k.shape)
print("V:", v.shape)


Q: torch.Size([2, 4, 8, 16])
K: torch.Size([2, 4, 8, 16])
V: torch.Size([2, 4, 8, 16])


## 10. 为什么计算 QK^T？

我们希望每一个 Query token 都和所有 Key token 计算相似度。

当前：

$$
Q.shape=(B,H,T,D_h)
$$

$$
K.shape=(B,H,T,D_h)
$$

如果直接计算：

$$
QK
$$

观察最后两个维度：

$$
(T,D_h)(T,D_h)
$$

中间维度：

$$
D_h
$$

和：

$$
T
$$

通常并不相等。

所以不能直接进行我们想要的矩阵乘法。

因此需要把 K 的最后两个维度交换：

$$
K^\top.shape=(B,H,D_h,T)
$$

也就是：

`K.transpose(-2, -1)`

于是：

$$
Q.shape=(B,H,T,D_h)
$$

$$
K^\top.shape=(B,H,D_h,T)
$$

现在最后两个矩阵维度满足：

$$
(T,D_h)(D_h,T)
$$

因此可以相乘。


In [9]:
B = 2
H = 4
T = 8
Dh = 16

q = torch.randn(B, H, T, Dh)
k = torch.randn(B, H, T, Dh)

k_t = k.transpose(-2, -1)

print("Q  :", q.shape)
print("K  :", k.shape)
print("K^T:", k_t.shape)


Q  : torch.Size([2, 4, 8, 16])
K  : torch.Size([2, 4, 8, 16])
K^T: torch.Size([2, 4, 16, 8])


## 11. Self-Attention 最重要的 Shape 推导

现在：

$$
Q.shape=(B,H,T,D_h)
$$

$$
K^\top.shape=(B,H,D_h,T)
$$

矩阵乘法：

$$
QK^\top
$$

只关注最后两个维度：

$$
(T,D_h)(D_h,T)
$$

中间的：

$$
D_h
$$

发生 contraction。

所以得到：

$$
(T,T)
$$

前面的 Batch 和 Head 维保持：

$$
(B,H)
$$

最终：

$$
QK^\top.shape=(B,H,T,T)
$$

完整写法：

$$
(B,H,T,D_h)
(B,H,D_h,T)
\rightarrow
(B,H,T,T)
$$

这就是 Attention Score Matrix 的 shape。


In [10]:
B = 2
H = 4
T = 8
Dh = 16

q = torch.randn(B, H, T, Dh)
k = torch.randn(B, H, T, Dh)

scores = q @ k.transpose(-2, -1)

print("Q     :", q.shape)
print("K     :", k.shape)
print("Scores:", scores.shape)


Q     : torch.Size([2, 4, 8, 16])
K     : torch.Size([2, 4, 8, 16])
Scores: torch.Size([2, 4, 8, 8])


## 12. Attention Score Matrix 的语义

为什么最终是：

$$
(T,T)
$$

而不是：

$$
(T,D_h)
$$

？

因为 Self-Attention 的目标是：

> 每一个 Query token 与每一个 Key token 比较。

假设：

$$
T=3
$$

那么 score matrix：

$$
S \in \mathbb{R}^{3\times3}
$$

可以理解为：

$$
S=
\begin{bmatrix}
q_1k_1 & q_1k_2 & q_1k_3 \\
q_2k_1 & q_2k_2 & q_2k_3 \\
q_3k_1 & q_3k_2 & q_3k_3
\end{bmatrix}
$$

其中：

- 行：Query token
- 列：Key token

因此：

$$
scores[...,i,j]
$$

表示：

> 第 i 个 Query token 对第 j 个 Key token 的 Attention Score。

所以：

$$
(T,T)
$$

本质上表示所有 token 两两之间的关系。


## 13. Scaled Dot-Product Attention

标准 Attention 不直接使用：

$$
QK^\top
$$

而是：

$$
\frac{QK^\top}{\sqrt{D_h}}
$$

因此：

$$
scores
=
\frac{QK^\top}{\sqrt{D_h}}
$$

这里除以：

$$
\sqrt{D_h}
$$

只是逐元素缩放。

所以 shape 不发生变化：

$$
(B,H,T,T)
\rightarrow
(B,H,T,T)
$$

为什么需要这个 scaling，我们后面正式学习 Attention 时再从方差和 Softmax 数值行为解释。

当前只需要记住：

> Scaling 不改变 shape。


In [11]:
import math

B = 2
H = 4
T = 8
Dh = 16

q = torch.randn(B, H, T, Dh)
k = torch.randn(B, H, T, Dh)

scores = q @ k.transpose(-2, -1)
scaled_scores = scores / math.sqrt(Dh)

print("scores       :", scores.shape)
print("scaled_scores:", scaled_scores.shape)


scores       : torch.Size([2, 4, 8, 8])
scaled_scores: torch.Size([2, 4, 8, 8])


## 14. Softmax 不改变 Shape

Attention 下一步会对最后一个维度做 Softmax：

$$
A
=
\operatorname{softmax}(scores)
$$

输入：

$$
scores.shape=(B,H,T,T)
$$

输出：

$$
A.shape=(B,H,T,T)
$$

Softmax 改变的是数值，而不是 Tensor shape。

通常：

`softmax(dim=-1)`

表示：

> 对每个 Query token，在所有 Key token 上计算一个概率分布。

因此最后一个维度上的元素满足：

$$
\sum_j A_{ij}=1
$$


In [12]:
B = 2
H = 4
T = 8
Dh = 16

q = torch.randn(B, H, T, Dh)
k = torch.randn(B, H, T, Dh)

scores = q @ k.transpose(-2, -1)

attention = torch.softmax(
    scores,
    dim=-1,
)

print("scores   :", scores.shape)
print("attention:", attention.shape)

print(
    "sum over keys:",
    attention[0, 0, 0].sum(),
)


scores   : torch.Size([2, 4, 8, 8])
attention: torch.Size([2, 4, 8, 8])
sum over keys: tensor(1.0000)


## 15. Attention Weights 与 V 相乘

现在：

$$
A.shape=(B,H,T,T)
$$

而：

$$
V.shape=(B,H,T,D_h)
$$

执行：

$$
AV
$$

只观察最后两个矩阵维度：

$$
(T,T)(T,D_h)
$$

中间的 T 被 contraction：

$$
(T,\cancel{T})(\cancel{T},D_h)
$$

最终：

$$
(T,D_h)
$$

加上前面的 Batch 和 Head：

$$
(B,H,T,D_h)
$$

所以：

$$
(B,H,T,T)
(B,H,T,D_h)
\rightarrow
(B,H,T,D_h)
$$

这是 Self-Attention 中第二个非常关键的矩阵乘法。


In [13]:
B = 2
H = 4
T = 8
Dh = 16

q = torch.randn(B, H, T, Dh)
k = torch.randn(B, H, T, Dh)
v = torch.randn(B, H, T, Dh)

scores = q @ k.transpose(-2, -1)

attention = torch.softmax(
    scores,
    dim=-1,
)

output = attention @ v

print("attention:", attention.shape)
print("V        :", v.shape)
print("output   :", output.shape)


attention: torch.Size([2, 4, 8, 8])
V        : torch.Size([2, 4, 8, 16])
output   : torch.Size([2, 4, 8, 16])


## 16. Self-Attention Shape Pipeline

目前我们已经能够从 shape 上看懂 Self-Attention 最核心的部分。

### Step 1：输入

$$
X.shape=(B,T,D)
$$

### Step 2：拆分 Attention Heads

$$
(B,T,D)
\rightarrow
(B,H,T,D_h)
$$

其中：

$$
D=HD_h
$$

### Step 3：Q 与 K 计算 Attention Scores

$$
Q.shape=(B,H,T,D_h)
$$

$$
K.shape=(B,H,T,D_h)
$$

转置 K：

$$
K^\top.shape=(B,H,D_h,T)
$$

于是：

$$
QK^\top
$$

得到：

$$
(B,H,T,T)
$$

### Step 4：Scaling

$$
\frac{QK^\top}{\sqrt{D_h}}
$$

shape 不变：

$$
(B,H,T,T)
$$

### Step 5：Softmax

$$
A=
\operatorname{softmax}
\left(
\frac{QK^\top}{\sqrt{D_h}}
\right)
$$

shape：

$$
(B,H,T,T)
$$

### Step 6：Attention × V

$$
A.shape=(B,H,T,T)
$$

$$
V.shape=(B,H,T,D_h)
$$

因此：

$$
AV.shape=(B,H,T,D_h)
$$


## 17. 从 Multi-Head 输出恢复 `(B,T,D)`

Attention 每个 Head 的输出为：

$$
(B,H,T,D_h)
$$

但 Transformer 后续通常希望恢复：

$$
(B,T,D)
$$

第一步交换 Head 和 Token：

$$
(B,H,T,D_h)
\rightarrow
(B,T,H,D_h)
$$

然后合并：

$$
H\times D_h=D
$$

因此：

$$
(B,T,H,D_h)
\rightarrow
(B,T,D)
$$

完整过程：

$$
(B,H,T,D_h)
\rightarrow
(B,T,H,D_h)
\rightarrow
(B,T,D)
$$


In [14]:
B = 2
H = 4
T = 8
Dh = 16

D = H * Dh

x = torch.randn(B, H, T, Dh)

x = x.transpose(1, 2)

print("After transpose:", x.shape)

x = x.reshape(B, T, D)

print("After reshape  :", x.shape)


After transpose: torch.Size([2, 8, 4, 16])
After reshape  : torch.Size([2, 8, 64])


In [15]:
B = 2
T = 8
D = 64
H = 4

Dh = D // H


q = torch.randn(B, T, D)
k = torch.randn(B, T, D)
v = torch.randn(B, T, D)


# (B,T,D) -> (B,T,H,Dh)
q = q.reshape(B, T, H, Dh)
k = k.reshape(B, T, H, Dh)
v = v.reshape(B, T, H, Dh)


# (B,T,H,Dh) -> (B,H,T,Dh)
q = q.transpose(1, 2)
k = k.transpose(1, 2)
v = v.transpose(1, 2)


print("Q:", q.shape)
print("K:", k.shape)
print("V:", v.shape)


# (B,H,T,Dh) @ (B,H,Dh,T)
# -> (B,H,T,T)
scores = q @ k.transpose(-2, -1)

scores = scores / math.sqrt(Dh)

print("Scores:", scores.shape)


attention = torch.softmax(
    scores,
    dim=-1,
)

print("Attention:", attention.shape)


# (B,H,T,T) @ (B,H,T,Dh)
# -> (B,H,T,Dh)
output = attention @ v

print("Head output:", output.shape)


# (B,H,T,Dh) -> (B,T,H,Dh)
output = output.transpose(1, 2)

# (B,T,H,Dh) -> (B,T,D)
output = output.reshape(B, T, D)

print("Final output:", output.shape)


Q: torch.Size([2, 4, 8, 16])
K: torch.Size([2, 4, 8, 16])
V: torch.Size([2, 4, 8, 16])
Scores: torch.Size([2, 4, 8, 8])
Attention: torch.Size([2, 4, 8, 8])
Head output: torch.Size([2, 4, 8, 16])
Final output: torch.Size([2, 8, 64])


## 19. Shape Thinking：第一次完整阅读 Attention

现在不要看具体数值，只看 Tensor shape：

$$
X
$$

$$
(B,T,D)
$$

拆 Head：

$$
(B,T,D)
\rightarrow
(B,H,T,D_h)
$$

QK：

$$
(B,H,T,D_h)
@
(B,H,D_h,T)
$$

得到：

$$
(B,H,T,T)
$$

Softmax：

$$
(B,H,T,T)
$$

再乘 V：

$$
(B,H,T,T)
@
(B,H,T,D_h)
$$

得到：

$$
(B,H,T,D_h)
$$

重新合并 Heads：

$$
(B,H,T,D_h)
\rightarrow
(B,T,D)
$$

因此从输入到输出：

$$
(B,T,D)
\rightarrow
(B,T,D)
$$

虽然最终 shape 没变，但内部经过了：

$$
T\times T
$$

的 Attention Matrix。

这就是 Transformer 能够让每个 token 与其它 token 交互的关键。


## 20. 常见错误

### 错误 1：把 `*` 当成 Matrix Multiplication

`A * B`

是 element-wise multiplication。

`A @ B`

才是 matrix multiplication。

---

### 错误 2：只看整个 Tensor，不看最后两个维度

对于高维 matmul：

$$
(...,M,K)(...,K,N)
\rightarrow
(...,M,N)
$$

首先看最后两个维度。

---

### 错误 3：忘记 transpose K

Q 和 K 都是：

$$
(B,H,T,D_h)
$$

不能直接得到我们想要的 token-to-token matrix。

需要：

$$
K^\top=(B,H,D_h,T)
$$

然后：

$$
QK^\top
\rightarrow
(B,H,T,T)
$$

---

### 错误 4：把 `(T,T)` 当成 feature dimension

Attention Score 中：

$$
(T,T)
$$

表示：

> Query Token × Key Token

不是 embedding dimension。

---

### 错误 5：认为 Softmax 会改变 shape

Softmax 只改变数值分布。

$$
(B,H,T,T)
\rightarrow
(B,H,T,T)
$$

---

### 错误 6：Attention × V 时 contraction 维度看错

$$
(B,H,T,T)
(B,H,T,D_h)
$$

最后两个矩阵：

$$
(T,T)(T,D_h)
$$

所以输出：

$$
(T,D_h)
$$

最终：

$$
(B,H,T,D_h)
$$


## 本节总结

### Rule 1：二维 Matrix Multiplication

$$
(M,K)(K,N)
\rightarrow
(M,N)
$$

中间的 K 被 contraction。

---

### Rule 2：Linear Projection

$$
(B,T,D)(D,H)
\rightarrow
(B,T,H)
$$

Linear Layer 相当于对所有 token 应用相同的线性变换。

---

### Rule 3：High-Dimensional Matmul

一般形式：

$$
(...,M,K)(...,K,N)
\rightarrow
(...,M,N)
$$

最后两个维度进行矩阵乘法。

前面的维度作为 batch dimensions。

---

### Rule 4：Attention QK^T

$$
Q=(B,H,T,D_h)
$$

$$
K=(B,H,T,D_h)
$$

转置：

$$
K^\top=(B,H,D_h,T)
$$

因此：

$$
QK^\top
=
(B,H,T,T)
$$

---

### Rule 5：Attention Score 的语义

$$
(T,T)
$$

表示：

$$
Query\ Token
\times
Key\ Token
$$

也就是所有 token 两两之间的 Attention Score。

---

### Rule 6：Attention × V

$$
(B,H,T,T)
(B,H,T,D_h)
\rightarrow
(B,H,T,D_h)
$$

---

### Rule 7：最终恢复 Model Dimension

$$
(B,H,T,D_h)
\rightarrow
(B,T,H,D_h)
\rightarrow
(B,T,D)
$$

其中：

$$
D=HD_h
$$
